# Mathemagicland 2D Random Walk Solution

This notebook simulates the 2D walk from mathemagicland's idea and stops at the first return to the origin.

Video solution reference:
[https://www.instagram.com/mathemagiclandinsta/reel/DW2BveQklDj/](https://www.instagram.com/mathemagiclandinsta/reel/DW2BveQklDj/)

Outputs are saved in `outputs/mathemagicland_solution/`.

In [1]:
from pathlib import Path
import json
import math
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configuration
RANDOM_SEED = 20260407
N_TRIALS = 300
MAX_STEPS = 20_000

NOTEBOOK_NAME = 'mathemagicland_solution.ipynb'

cwd = Path.cwd()
if (cwd / NOTEBOOK_NAME).exists():
    NOTEBOOK_DIR = cwd
elif (cwd / 'random_walks' / NOTEBOOK_NAME).exists():
    NOTEBOOK_DIR = cwd / 'random_walks'
else:
    NOTEBOOK_DIR = cwd

OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / 'mathemagicland_solution'
PATHS_DIR = OUTPUT_DIR / 'paths'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PATHS_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
rng = np.random.default_rng(RANDOM_SEED)

print(f'Notebook directory: {NOTEBOOK_DIR}')
print(f'Output directory:   {OUTPUT_DIR}')

Notebook directory: /home/manpazito/projects/fun_mini_sims/random_walks
Output directory:   /home/manpazito/projects/fun_mini_sims/random_walks/outputs/mathemagicland_solution


## Mathematical Description

Each step is chosen uniformly from four cardinal directions:
- 0 degrees   -> (1, 0)
- 90 degrees  -> (0, 1)
- 180 degrees -> (-1, 0)
- 270 degrees -> (0, -1)

Starting point is (0, 0). We stop at the first n >= 1 such that the walk returns to (0, 0).

As in the other notebooks, we use `MAX_STEPS` to handle long trajectories and label non-returning trials as censored.

In [2]:
CARDINAL_STEPS = np.array(
    [
        [1, 0],
        [0, 1],
        [-1, 0],
        [0, -1],
    ],
    dtype=int,
)


def simulate_mathemagic_walk(rng: np.random.Generator, max_steps: int) -> Dict[str, object]:
    # Simulate one 2D cardinal random walk until first return or max_steps.
    x, y = 0, 0
    xs: List[int] = [x]
    ys: List[int] = [y]

    min_x = max_x = x
    min_y = max_y = y
    max_distance = 0.0
    max_manhattan = 0
    returned_to_origin = False

    for _ in range(max_steps):
        dx, dy = CARDINAL_STEPS[int(rng.integers(0, 4))]
        x += int(dx)
        y += int(dy)
        xs.append(x)
        ys.append(y)

        min_x = min(min_x, x)
        max_x = max(max_x, x)
        min_y = min(min_y, y)
        max_y = max(max_y, y)

        distance = math.hypot(x, y)
        max_distance = max(max_distance, distance)
        max_manhattan = max(max_manhattan, abs(x) + abs(y))

        if x == 0 and y == 0:
            returned_to_origin = True
            break

    steps_until_stop = len(xs) - 1
    return_time = float(steps_until_stop) if returned_to_origin else np.nan
    bbox_width = max_x - min_x
    bbox_height = max_y - min_y

    return {
        'xs': xs,
        'ys': ys,
        'steps_until_stop': steps_until_stop,
        'returned_to_origin': returned_to_origin,
        'censored': not returned_to_origin,
        'return_time': return_time,
        'max_distance': float(max_distance),
        'max_manhattan': int(max_manhattan),
        'bbox_width': int(bbox_width),
        'bbox_height': int(bbox_height),
        'bbox_area': int(bbox_width * bbox_height),
        'final_x': int(x),
        'final_y': int(y),
    }


def save_path_json(xs: List[int], ys: List[int], trial_index: int, paths_dir: Path) -> Path:
    payload = {
        'trial_index': int(trial_index),
        'x': [int(v) for v in xs],
        'y': [int(v) for v in ys],
    }
    out_path = paths_dir / f'trial_{trial_index:04d}.json'
    with out_path.open('w', encoding='utf-8') as f:
        json.dump(payload, f)
    return out_path


def plot_single_walk(xs: List[int], ys: List[int], out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    ax.plot(xs, ys, color='tab:blue', linewidth=2, alpha=0.95)
    ax.scatter([0], [0], color='green', marker='*', s=180, label='origin')
    ax.scatter([xs[-1]], [ys[-1]], color='red', marker='X', s=120, label='stop point')
    ax.set_title('Single 2D Walk (Mathemagicland Model)')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal', adjustable='box')
    ax.legend(loc='best')
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_return_histogram(summary_df: pd.DataFrame, out_path: Path) -> None:
    completed = summary_df.loc[~summary_df['censored'], 'return_time']
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.hist(completed, bins=30, color='tab:blue', edgecolor='white')
    ax.set_title('Return Time Histogram (Completed Trials)')
    ax.set_xlabel('Steps until first return')
    ax.set_ylabel('Count')
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_max_distance_histogram(summary_df: pd.DataFrame, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.hist(summary_df['max_distance'], bins=30, color='tab:orange', edgecolor='white')
    ax.set_title('Max Euclidean Distance Histogram')
    ax.set_xlabel('max Euclidean distance during trial')
    ax.set_ylabel('Count')
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_overlay_walks(all_paths: List[Tuple[List[int], List[int]]], out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(7, 7))
    for xs, ys in all_paths:
        ax.plot(xs, ys, color='tab:blue', alpha=0.11, linewidth=0.8)

    ax.scatter([0], [0], color='black', marker='*', s=160, label='origin')
    ax.set_title('Overlay of All 2D Walks (Mathemagicland Model)')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal', adjustable='box')
    ax.legend(loc='best')
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

## Single Example Walk

Simulate one trajectory and plot it in 2D.

In [3]:
single_walk = simulate_mathemagic_walk(rng=rng, max_steps=MAX_STEPS)

single_walk_plot_path = OUTPUT_DIR / 'single_walk.png'
plot_single_walk(single_walk['xs'], single_walk['ys'], single_walk_plot_path)

print('Single walk summary:')
print(f"  steps_until_stop:   {single_walk['steps_until_stop']}")
print(f"  returned_to_origin: {single_walk['returned_to_origin']}")
print(f"  max_distance:       {single_walk['max_distance']:.3f}")
print(f"  bbox (w x h):       {single_walk['bbox_width']} x {single_walk['bbox_height']}")
print(f'  saved plot:         {single_walk_plot_path}')

Single walk summary:
  steps_until_stop:   2
  returned_to_origin: True
  max_distance:       1.000
  bbox (w x h):       1 x 0
  saved plot:         /home/manpazito/projects/fun_mini_sims/random_walks/outputs/mathemagicland_solution/single_walk.png


## Monte Carlo Simulation

Run many independent trials, save per-trial raw paths, and write summary statistics to CSV.

In [4]:
trial_records: List[Dict[str, object]] = []
all_paths: List[Tuple[List[int], List[int]]] = []

for trial_idx in range(N_TRIALS):
    walk = simulate_mathemagic_walk(rng=rng, max_steps=MAX_STEPS)
    all_paths.append((walk['xs'], walk['ys']))
    save_path_json(walk['xs'], walk['ys'], trial_idx, PATHS_DIR)

    trial_records.append(
        {
            'trial_index': trial_idx,
            'steps_until_stop': walk['steps_until_stop'],
            'return_time': walk['return_time'],
            'returned_to_origin': walk['returned_to_origin'],
            'censored': walk['censored'],
            'max_distance': walk['max_distance'],
            'max_manhattan': walk['max_manhattan'],
            'bbox_width': walk['bbox_width'],
            'bbox_height': walk['bbox_height'],
            'bbox_area': walk['bbox_area'],
            'final_x': walk['final_x'],
            'final_y': walk['final_y'],
        }
    )

summary_df = pd.DataFrame(trial_records)
summary_csv_path = OUTPUT_DIR / 'summary.csv'
summary_df.to_csv(summary_csv_path, index=False)

completed = summary_df[~summary_df['censored']]

print(f'Trials run:                {N_TRIALS}')
print(f'Completed returns:         {len(completed)}')
print(f'Censored trials:           {int(summary_df["censored"].sum())}')
if len(completed):
    print(f"Mean return time:          {completed['return_time'].mean():.3f}")
print(f"Mean max distance:         {summary_df['max_distance'].mean():.3f}")
print(f'Saved summary CSV:         {summary_csv_path}')

summary_df.head()

Trials run:                300
Completed returns:         230
Censored trials:           70
Mean return time:          881.730
Mean max distance:         47.466
Saved summary CSV:         /home/manpazito/projects/fun_mini_sims/random_walks/outputs/mathemagicland_solution/summary.csv


,trial_index,steps_until_stop,return_time,returned_to_origin,censored,max_distance,max_manhattan,bbox_width,bbox_height,bbox_area,final_x,final_y
0,0,20000,NaN,False,True,179.493732,250,202,118,23836,149,-91
1,1,20000,NaN,False,True,205.827598,283,181,189,34209,55,163
2,2,6,6.0,True,False,1.414214,2,2,1,2,0,0
3,3,20000,NaN,False,True,130.176803,184,144,128,18432,20,80
4,4,20000,NaN,False,True,185.528973,209,141,202,28482,-54,-130


## Visualization of Monte Carlo Results

Histograms for return times and max Euclidean distances.

In [5]:
return_hist_path = OUTPUT_DIR / 'return_time_histogram.png'
max_dist_hist_path = OUTPUT_DIR / 'max_distance_histogram.png'

plot_return_histogram(summary_df, return_hist_path)
plot_max_distance_histogram(summary_df, max_dist_hist_path)

print(f'Saved: {return_hist_path}')
print(f'Saved: {max_dist_hist_path}')

Saved: /home/manpazito/projects/fun_mini_sims/random_walks/outputs/mathemagicland_solution/return_time_histogram.png
Saved: /home/manpazito/projects/fun_mini_sims/random_walks/outputs/mathemagicland_solution/max_distance_histogram.png


## Final Overlay Plot

Overlay all simulated 2D paths with transparency and equal aspect ratio.

In [6]:
overlay_path = OUTPUT_DIR / 'overlay_walks.png'
plot_overlay_walks(all_paths, overlay_path)
print(f'Saved: {overlay_path}')

Saved: /home/manpazito/projects/fun_mini_sims/random_walks/outputs/mathemagicland_solution/overlay_walks.png
